# ONNX Runtime INT8 양자화 실습 (LLM: Causal LM)

> 강의/실습용 노트북 (업데이트: 2026-03-06)

이 노트북은 **작은 LLM(예: `distilgpt2`)을 PyTorch → ONNX로 Export**한 뒤,  
ONNX Runtime(ORT)에서 INT8 양자화(Dynamic Quantization 중심)를 적용하여

- ✅ 모델 파일 크기 변화  
- ✅ 추론 속도(지연시간) 변화  
- ✅ 출력/로그릿 차이 및 간단한 Perplexity 변화

를 **정성 + 정량**으로 확인하는 실습입니다.

---

## 오늘의 학습 목표

1. Causal LM을 ONNX로 export할 때 **입력/출력, dynamic axes**를 설정할 수 있다.
2. ORT 세션으로 ONNX 모델을 실행하고, PyTorch 결과와 **sanity check**를 수행할 수 있다.
3. ORT의 **Dynamic INT8 Quantization**을 적용하고 **속도/크기/품질** 트레이드오프를 측정할 수 있다.
4. (선택) Calibration을 이용한 **Static Quantization(QDQ)** 흐름을 이해한다.

---

## 주의사항 / 제한

- 이 노트북은 **교육 목적**으로 “가장 단순한 형태”의 ONNX export를 사용합니다.
- 여기서는 `past_key_values`를 export하지 않기 때문에, 생성(Generation)을 ORT로 하면 **매 step마다 전체 시퀀스를 다시 forward**합니다.  
  → 길이가 길어지면 O(n²)로 느려질 수 있어요.  
  실무에서는 `past_key_values` 포함 export, ORT Transformers, TensorRT, vLLM 같은 방식이 더 적합합니다.
- 모델은 기본값으로 `distilgpt2`(작고 다운로드 쉬움)를 사용합니다.  
  큰 모델은 export/양자화 시간이 훨씬 오래 걸릴 수 있습니다.

## 0. 환경 준비

아래 셀은 실습 환경에 필요한 패키지를 설치합니다.

- `transformers`: 모델/토크나이저 로딩
- `torch`: PyTorch 모델 실행 및 ONNX export
- `onnx`: ONNX 모델 로드/검사
- `onnxruntime`: ORT 세션 실행 + 양자화 도구(quantization)

> 이미 설치되어 있다면 설치 셀은 건너뛰어도 됩니다.

In [1]:
# (선택) 패키지 설치
# 로컬 환경에 따라 torch 설치는 시간이 오래 걸릴 수 있습니다.
# 필요 시 주석을 해제하세요.

%pip -q install -U onnx onnxruntime onnxscript
# %pip -q install -U "torch>=2.1" --index-url https://download.pytorch.org/whl/cpu

Note: you may need to restart the kernel to use updated packages.


## 1. 실습 설정

- 기본 모델: `distilgpt2` (작고 ONNX 실습에 적합)
- 출력 디렉토리: `./onnx_llm_artifacts`

In [14]:
import os
import time
import json
import numpy as np

MODEL_ID = os.environ.get("MODEL_ID", "distilgpt2")
ARTIFACT_DIR = os.environ.get("ARTIFACT_DIR", "./onnx_llm_artifacts")
os.makedirs(ARTIFACT_DIR, exist_ok=True)

FP32_ONNX_PATH = os.path.join(ARTIFACT_DIR, "quantize_fp32.onnx")
INT8_DQ_ONNX_PATH = os.path.join(ARTIFACT_DIR, "quantize_int8_dynamic.onnx")
INT8_SQ_ONNX_PATH = os.path.join(ARTIFACT_DIR, "quantize_int8_static_qdq.onnx")

SEED = 42
np.random.seed(SEED)

print("MODEL_ID:", MODEL_ID)
print("ARTIFACT_DIR:", ARTIFACT_DIR)

MODEL_ID: distilgpt2
ARTIFACT_DIR: ./onnx_llm_artifacts


## 2. PyTorch 모델/토크나이저 로딩

Causal LM(다음 토큰 예측) 모델을 로딩합니다.

- `AutoTokenizer`: 텍스트 → 토큰 id
- `AutoModelForCausalLM`: logits 출력 (shape: `[B, T, V]`)

In [15]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

device = "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID)
model.to(device)
model.eval()
model.config.use_cache = False

# GPT-2 계열은 pad_token이 없어서, 편의상 eos_token을 pad_token으로 사용합니다.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("vocab_size:", tokenizer.vocab_size)
print("pad_token:", tokenizer.pad_token, tokenizer.pad_token_id)

vocab_size: 50257
pad_token: <|endoftext|> 50256


## 3. 샘플 입력 만들기

ONNX export와 sanity check에 사용할 입력을 준비합니다.

- `input_ids`: `[B, T]` int64
- `attention_mask`: `[B, T]` int64 (0/1)

In [16]:
def make_batch(texts, max_length=32):
    enc = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=max_length,
    )
    input_ids = enc["input_ids"].to(device)
    attention_mask = enc["attention_mask"].to(device)
    return input_ids, attention_mask

texts = [
    "Today is a beautiful day and I want to",
    "In the future, AI systems will",
]
input_ids, attention_mask = make_batch(texts, max_length=32)

print("input_ids shape:", tuple(input_ids.shape), input_ids.dtype)
print("attention_mask shape:", tuple(attention_mask.shape), attention_mask.dtype)

input_ids shape: (2, 9) torch.int64
attention_mask shape: (2, 9) torch.int64


## 4. PyTorch → ONNX Export

### 왜 wrapper가 필요한가?
Transformers 모델은 출력이 `ModelOutput`(dict-like) 형태일 수 있습니다.  
ONNX로 내보낼 때는 출력 텐서를 명확히 지정하는 게 안전합니다.

여기서는 `logits`만 반환하는 wrapper를 만들고 export합니다.

### Dynamic Axes
- batch 크기(B)와 시퀀스 길이(T)가 바뀌어도 ONNX 모델이 동작하도록
`dynamic_axes`를 설정합니다.

In [17]:
import onnx
import torch

model.config.use_cache = False

class CausalLMWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, input_ids, attention_mask):
        out = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False,
            return_dict=True,
        )
        return out.logits

wrapper = CausalLMWrapper(model).to(device).eval()

def export_to_onnx(fp32_path: str, input_ids: torch.Tensor, attention_mask: torch.Tensor, opset=18):
    input_ids_cpu = input_ids.detach().cpu()
    attention_mask_cpu = attention_mask.detach().cpu()

    with torch.no_grad():
        torch.onnx.export(
            wrapper,
            (input_ids_cpu, attention_mask_cpu),
            fp32_path,
            input_names=["input_ids", "attention_mask"],
            output_names=["logits"],
            dynamic_axes={
                "input_ids": {0: "batch", 1: "seq"},
                "attention_mask": {0: "batch", 1: "seq"},
                "logits": {0: "batch", 1: "seq"},
            },
            opset_version=opset,
            do_constant_folding=True,
        )

    onnx_model = onnx.load(fp32_path)
    onnx.checker.check_model(onnx_model)
    return fp32_path

export_to_onnx(FP32_ONNX_PATH, input_ids, attention_mask)
print("Saved:", FP32_ONNX_PATH, "size(MB)=", os.path.getsize(FP32_ONNX_PATH) / 1024 / 1024)


/var/tmp/ipykernel_1237666/991660112.py:27: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0313 18:17:51.375000 1237666 site-packages/torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0313 18:17:51.377000 1237666 site-packages/torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0313 18:17:51.379000 1237666 site-packages/torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes,

[torch.onnx] Obtain model graph for `CausalLMWrapper([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `CausalLMWrapper([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


/opt/conda/envs/sam/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Applied 134 of general pattern rewrite rules.
Saved: ./onnx_llm_artifacts/quantize_fp32.onnx size(MB)= 0.6543550491333008


## 5. ORT에서 FP32 ONNX 실행 & PyTorch와 비교

- 같은 입력에 대해 logits가 거의 비슷하게 나오는지 확인합니다.
- ONNX export가 제대로 되었는지 sanity check로 매우 중요합니다.

In [18]:
import onnxruntime as ort

def run_pytorch_logits(input_ids: torch.Tensor, attention_mask: torch.Tensor) -> np.ndarray:
    with torch.no_grad():
        out = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = out.logits.detach().cpu().numpy()
    return logits

def run_ort_logits(session: ort.InferenceSession, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> np.ndarray:
    inputs = {
        "input_ids": input_ids.detach().cpu().numpy().astype(np.int64),
        "attention_mask": attention_mask.detach().cpu().numpy().astype(np.int64),
    }
    (logits,) = session.run(["logits"], inputs)
    return logits

sess_options = ort.SessionOptions()
sess_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
# CPU thread 설정(선택): sess_options.intra_op_num_threads = os.cpu_count()
sess_fp32 = ort.InferenceSession(FP32_ONNX_PATH, sess_options=sess_options, providers=["CPUExecutionProvider"])

pt_logits = run_pytorch_logits(input_ids, attention_mask)
ort_logits = run_ort_logits(sess_fp32, input_ids, attention_mask)

diff = np.max(np.abs(pt_logits - ort_logits))
print("max_abs_diff:", diff)

max_abs_diff: 0.00010681152


## 6. (보너스) ORT로 간단한 Greedy Generation 해보기

> 주의: 이 방식은 `past_key_values`를 쓰지 않아서 길이가 길어지면 느립니다.  
"ONNX 모델로 생성" 하는 것을 가볍게 확인합니다.

In [19]:
def greedy_generate_ort(session, prompt: str, max_new_tokens=20, max_length=64):
    # 1) 프롬프트 토크나이즈
    enc = tokenizer(prompt, return_tensors="pt")
    input_ids = enc["input_ids"].to(device)
    attention_mask = enc["attention_mask"].to(device)

    # 2) 반복 생성
    for _ in range(max_new_tokens):
        logits = run_ort_logits(session, input_ids, attention_mask)
        next_token_logits = logits[:, -1, :]  # [B, V]
        next_token_id = int(np.argmax(next_token_logits, axis=-1)[0])

        # append
        next_token = torch.tensor([[next_token_id]], dtype=input_ids.dtype)
        input_ids = torch.cat([input_ids, next_token], dim=1)
        attention_mask = torch.cat([attention_mask, torch.ones_like(next_token)], dim=1)

        if input_ids.shape[1] >= max_length:
            break

    return tokenizer.decode(input_ids[0], skip_special_tokens=True)

print(greedy_generate_ort(sess_fp32, "Q: What is ONNX?\nA:", max_new_tokens=40))

Q: What is ONNX?
A: ONNX is a software program that allows you to create a program that can be used to create a program that can be used to create a program that can be used to create a program that can


## 7. 성능 측정(Forward Latency)

여기서는 생성(step-by-step)이 아닌, **고정 길이 입력에 대한 forward 1회 latency**를 측정합니다.

- warmup을 몇 회 돌리고
- 평균/표준편차/대략적인 p95를 계산합니다.

In [20]:
def benchmark_session(session, batch_size=1, seq_len=32, iters=50, warmup=10):
    # 더미 입력
    dummy_texts = ["Hello world!"] * batch_size
    input_ids, attention_mask = make_batch(dummy_texts, max_length=seq_len)

    # warmup
    for _ in range(warmup):
        _ = run_ort_logits(session, input_ids, attention_mask)

    times = []
    for _ in range(iters):
        t0 = time.perf_counter()
        _ = run_ort_logits(session, input_ids, attention_mask)
        t1 = time.perf_counter()
        times.append((t1 - t0) * 1000.0)  # ms

    times = np.array(times)
    return {
        "mean_ms": float(times.mean()),
        "std_ms": float(times.std()),
        "p95_ms": float(np.percentile(times, 95)),
    }

fp32_b1 = benchmark_session(sess_fp32, batch_size=1, seq_len=32, iters=50, warmup=10)
fp32_b8 = benchmark_session(sess_fp32, batch_size=8, seq_len=32, iters=50, warmup=10)
print("FP32 b1:", fp32_b1)
print("FP32 b8:", fp32_b8)

FP32 b1: {'mean_ms': 26.12970738147851, 'std_ms': 0.6850417424830181, 'p95_ms': 27.10169015044812}
FP32 b8: {'mean_ms': 56.77284326287918, 'std_ms': 1.1299700648126347, 'p95_ms': 58.43152900488349}


## 8. Dynamic INT8 Quantization 적용

Dynamic Quantization은 **캘리브레이션 데이터 없이**(즉시) 적용할 수 있고,  
주로 **가중치(weight) 중심**으로 INT8 변환을 수행합니다.

- Transformer/LLM에서는 `MatMul/Gemm`에 효과가 많이 나타나는 편입니다.

In [21]:
from onnxruntime.quantization import quantize_dynamic, QuantType

def dynamic_quantize(fp32_path: str, int8_path: str):
    quantize_dynamic(
        model_input=fp32_path,
        model_output=int8_path,
        weight_type=QuantType.QInt8,
        op_types_to_quantize=["MatMul", "Gemm"],
    )
    return int8_path

dynamic_quantize(FP32_ONNX_PATH, INT8_DQ_ONNX_PATH)
print("Saved:", INT8_DQ_ONNX_PATH, "size(MB)=", os.path.getsize(INT8_DQ_ONNX_PATH)/1024/1024)

Saved: ./onnx_llm_artifacts/quantize_int8_dynamic.onnx size(MB)= 191.59346389770508


## 9. INT8(Dynamic) 모델 검증 + 성능 비교

- FP32 vs INT8 logits 차이를 확인합니다.
- latency(평균/p95)를 비교합니다.

In [22]:
sess_int8_dq = ort.InferenceSession(INT8_DQ_ONNX_PATH, sess_options=sess_options, providers=["CPUExecutionProvider"])

ort_logits_int8 = run_ort_logits(sess_int8_dq, input_ids, attention_mask)
diff_int8 = np.max(np.abs(ort_logits - ort_logits_int8))
print("FP32(ORT) vs INT8(DQ) max_abs_diff:", diff_int8)

int8_b1 = benchmark_session(sess_int8_dq, batch_size=1, seq_len=32, iters=50, warmup=10)
int8_b8 = benchmark_session(sess_int8_dq, batch_size=8, seq_len=32, iters=50, warmup=10)
print("INT8(DQ) b1:", int8_b1)
print("INT8(DQ) b8:", int8_b8)

FP32(ORT) vs INT8(DQ) max_abs_diff: 10.386757
INT8(DQ) b1: {'mean_ms': 10.732062915922143, 'std_ms': 0.8888271974214168, 'p95_ms': 12.472221690404695}
INT8(DQ) b8: {'mean_ms': 29.140733358217403, 'std_ms': 2.995781256719259, 'p95_ms': 34.292730248125736}


## 10. (정량) 간단한 Perplexity 비교

Perplexity는 “토큰 예측 난이도”를 간단히 요약해주는 지표입니다.

여기서는 아주 작은 텍스트 샘플로만 평가합니다(교육용).  
FP32 / INT8(Dynamic)에서 perplexity가 얼마나 달라지는지 봅니다.

> 주의: 샘플이 작으면 분산이 커서 결과가 불안정할 수 있습니다.

In [23]:
def compute_ppl_from_logits(logits: np.ndarray, input_ids: np.ndarray, attention_mask: np.ndarray) -> float:
    # logits: [B, T, V], input_ids: [B, T]
    # next-token prediction: shift
    logits = logits[:, :-1, :]           # predict token t+1
    labels = input_ids[:, 1:]            # actual token t+1
    mask = attention_mask[:, 1:]         # ignore padding

    # log-softmax
    log_probs = logits - np.log(np.sum(np.exp(logits - logits.max(axis=-1, keepdims=True)), axis=-1, keepdims=True)) - logits.max(axis=-1, keepdims=True)
    # pick label prob
    B, Tm1 = labels.shape
    idx_b = np.arange(B)[:, None]
    idx_t = np.arange(Tm1)[None, :]
    chosen = log_probs[idx_b, idx_t, labels]

    # average negative log likelihood over valid tokens
    nll = -chosen * mask
    denom = np.maximum(mask.sum(), 1.0)
    mean_nll = nll.sum() / denom
    return float(np.exp(mean_nll))

def compute_ppl_ort(session, texts, max_length=64):
    input_ids_t, attention_mask_t = make_batch(texts, max_length=max_length)
    logits = run_ort_logits(session, input_ids_t, attention_mask_t)
    ppl = compute_ppl_from_logits(
        logits,
        input_ids_t.detach().cpu().numpy(),
        attention_mask_t.detach().cpu().numpy(),
    )
    return ppl

eval_texts = [
    "The quick brown fox jumps over the lazy dog.",
    "Deep learning models can be exported to ONNX for deployment.",
    "Quantization often improves CPU inference speed.",
    "Transformers use attention mechanisms to process sequences.",
]

ppl_fp32 = compute_ppl_ort(sess_fp32, eval_texts, max_length=64)
ppl_int8 = compute_ppl_ort(sess_int8_dq, eval_texts, max_length=64)
print("PPL FP32:", ppl_fp32)
print("PPL INT8(DQ):", ppl_int8)

PPL FP32: 575.5497355403387
PPL INT8(DQ): 649.5149152072538


## 11. 결과 요약

- 파일 크기
- latency
- ppl

를 한 표로 정리합니다.

In [24]:
from pathlib import Path

def mb(path):
    p = Path(path)
    total = p.stat().st_size
    sidecar = Path(str(p) + ".data")
    if sidecar.exists():
        total += sidecar.stat().st_size
    return total / 1024 / 1024

summary = {
    "model": MODEL_ID,
    "fp32_onnx_mb": mb(FP32_ONNX_PATH),
    "int8_dynamic_onnx_mb": mb(INT8_DQ_ONNX_PATH),
    "fp32_latency_b1_ms": fp32_b1["mean_ms"],
    "int8_latency_b1_ms": int8_b1["mean_ms"],
    "fp32_latency_b8_ms": fp32_b8["mean_ms"],
    "int8_latency_b8_ms": int8_b8["mean_ms"],
    "ppl_fp32": ppl_fp32,
    "ppl_int8_dynamic": ppl_int8,
}
summary

{'model': 'distilgpt2',
 'fp32_onnx_mb': 313.1416597366333,
 'int8_dynamic_onnx_mb': 191.59346389770508,
 'fp32_latency_b1_ms': 26.12970738147851,
 'int8_latency_b1_ms': 10.732062915922143,
 'fp32_latency_b8_ms': 56.77284326287918,
 'int8_latency_b8_ms': 29.140733358217403,
 'ppl_fp32': 575.5497355403387,
 'ppl_int8_dynamic': 649.5149152072538}

## 12. (선택) Static Quantization(QDQ) — Calibration 기반

Static Quantization은 calibration 데이터로 activation 범위를 추정합니다.  
LLM은 그래프 구조가 복잡해서 환경/버전에 따라 quantize_static이 실패할 수도 있어요.

따라서 여기서는 “도전 과제(Challenge)”로 제공합니다.

### 실습 포인트
- calibration 데이터의 품질/양이 결과에 영향을 줌
- QDQ(QuantizeLinear/DequantizeLinear) 형태로 삽입

In [25]:
from onnxruntime.quantization import quantize_static, CalibrationDataReader, QuantFormat, QuantType

class SimpleCalibrationDataReader(CalibrationDataReader):
    def __init__(self, texts, max_length=32, batch_size=1):
        self.texts = texts
        self.max_length = max_length
        self.batch_size = batch_size
        self._iter = None

    def get_next(self):
        if self._iter is None:
            # 미리 전처리해서 list로 만들어둠
            batches = []
            for i in range(0, len(self.texts), self.batch_size):
                batch_texts = self.texts[i:i+self.batch_size]
                input_ids, attention_mask = make_batch(batch_texts, max_length=self.max_length)
                batches.append({
                    "input_ids": input_ids.detach().cpu().numpy().astype(np.int64),
                    "attention_mask": attention_mask.detach().cpu().numpy().astype(np.int64),
                })
            self._iter = iter(batches)
        return next(self._iter, None)

static_quant_ok = False

calib_texts = [
    "Calibration data should represent typical inference inputs.",
    "Quantization range estimation affects accuracy and stability.",
    "We measure latency and perplexity for quantized models.",
] * 4

try:
    dr = SimpleCalibrationDataReader(calib_texts, max_length=32, batch_size=1)
    quantize_static(
        model_input=FP32_ONNX_PATH,
        model_output=INT8_SQ_ONNX_PATH,
        calibration_data_reader=dr,
        quant_format=QuantFormat.QDQ,
        activation_type=QuantType.QInt8,
        weight_type=QuantType.QInt8,
        op_types_to_quantize=["MatMul", "Gemm"],
        per_channel=True,
    )
    static_quant_ok = True
    print("Saved:", INT8_SQ_ONNX_PATH, "size(MB)=", mb(INT8_SQ_ONNX_PATH))
except Exception as e:
    print("Static quantization failed (환경/버전/그래프 호환성 이슈일 수 있음):")
    print(type(e).__name__, e)

Saved: ./onnx_llm_artifacts/quantize_int8_static_qdq.onnx size(MB)= 192.18790817260742


## 13. 최종 비교: FP32 vs INT8(Dynamic) vs INT8(Static)

Static quantization이 성공했다면 세 모델의 크기, latency, perplexity를 한 번에 비교합니다.

In [26]:
import pandas as pd

final_rows = [
    {
        "variant": "fp32",
        "onnx_mb": mb(FP32_ONNX_PATH),
        "latency_b1_ms": fp32_b1["mean_ms"],
        "latency_b8_ms": fp32_b8["mean_ms"],
        "ppl": ppl_fp32,
    },
    {
        "variant": "int8_dynamic",
        "onnx_mb": mb(INT8_DQ_ONNX_PATH),
        "latency_b1_ms": int8_b1["mean_ms"],
        "latency_b8_ms": int8_b8["mean_ms"],
        "ppl": ppl_int8,
    },
]

static_ready = static_quant_ok and os.path.exists(INT8_SQ_ONNX_PATH)
if static_ready:
    sess_int8_sq = ort.InferenceSession(
        INT8_SQ_ONNX_PATH,
        sess_options=sess_options,
        providers=["CPUExecutionProvider"],
    )
    int8_sq_b1 = benchmark_session(sess_int8_sq, batch_size=1, seq_len=32, iters=50, warmup=10)
    int8_sq_b8 = benchmark_session(sess_int8_sq, batch_size=8, seq_len=32, iters=50, warmup=10)
    ppl_int8_sq = compute_ppl_ort(sess_int8_sq, eval_texts, max_length=64)
    final_rows.append({
        "variant": "int8_static_qdq",
        "onnx_mb": mb(INT8_SQ_ONNX_PATH),
        "latency_b1_ms": int8_sq_b1["mean_ms"],
        "latency_b8_ms": int8_sq_b8["mean_ms"],
        "ppl": ppl_int8_sq,
    })
else:
    print("Static quantized model not found. Skipping static comparison.")

final_compare = pd.DataFrame(final_rows)
final_compare["size_reduction_vs_fp32_%"] = (1 - final_compare["onnx_mb"] / final_compare.loc[final_compare["variant"] == "fp32", "onnx_mb"].iloc[0]) * 100
final_compare["speedup_b1_vs_fp32"] = final_compare.loc[final_compare["variant"] == "fp32", "latency_b1_ms"].iloc[0] / final_compare["latency_b1_ms"]
final_compare["speedup_b8_vs_fp32"] = final_compare.loc[final_compare["variant"] == "fp32", "latency_b8_ms"].iloc[0] / final_compare["latency_b8_ms"]
final_compare

,variant,onnx_mb,latency_b1_ms,latency_b8_ms,ppl,size_reduction_vs_fp32_%,speedup_b1_vs_fp32,speedup_b8_vs_fp32
0,fp32,313.141660,26.129707,56.772843,575.549736,0.000000,1.000000,1.000000
1,int8_dynamic,191.593464,10.732063,29.140733,649.514915,38.815722,2.434733,1.948230
2,int8_static_qdq,192.187908,6.182365,26.680000,1092.230878,38.625890,4.226491,2.127918


## 14. 과제(Exercises)

1) `MODEL_ID`를 `gpt2`로 바꾸고 다시 실행해보세요.  
   - export 시간 / 파일 크기 / latency가 어떻게 변하나요?

2) Dynamic Quantization에서 `op_types_to_quantize`를 변경해보세요.  
   - `["MatMul"]`만 적용 vs `["MatMul","Gemm"]` 적용 비교

3) (고급) `past_key_values`를 포함해서 export하면 generation 성능이 왜 개선될까요?  
   - 어떤 입력/출력 시그니처가 필요할지 설계해보세요.